# SQL practice

## Data Flow
 - data/raw/inpatient_claims.csv : Synthetic Data generated by "generate_demo_data.py"
 - data/processed/inpatient_claims_clean.csv : cleaned data by "validataion.py + transform.py"
 - data/processed/provider_features.csv: provider-wise feature summary by sql_features.py + provider_features.sql

In [29]:
import sqlite3
from pathlib import Path

import pandas as pd

root = Path.cwd()

print(f"current working directory: {root}")

if not (root / "config" / "config.yaml").exists():
    root = root.parent

db_path = root / "artifacts" / "claims.db"
db_path.parent.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(db_path)                  # database connection

current working directory: /home/youy/PROJECTS/MLOps_demos/cms-claims-anomaly-mlops/practice


## SQL 1: claims count by provider

In [16]:
# SQL 1: claims count by provider
query_claims_count = pd.read_sql_query(
    """
    SELECT COUNT(*) AS claim_count
    FROM inpatient_claims
    """,
    conn,
)

print(query_claims_count)

   claim_count
0        12000


## SQL 2: provider 별 claim 수 조회

In [ ]:

query_claims_count_by_provider = pd.read_sql_query(
    """
    SELECT 
        PRVDR_NUM, 
        COUNT(*) AS claim_count
    FROM inpatient_claims
    GROUP BY PRVDR_NUM
    ORDER BY claim_count DESC
    LIMIT 10
    """,
    conn,
)

print(query_claims_count_by_provider)

  PRVDR_NUM  claim_count
0     P0115          353
1     P0120          351
2     P0119          347
3     P0116          346
4     P0118          331
5     P0117          321
6     P0091          111
7     P0083          106
8     P0055          106
9     P0077          103


## SQL 3: provider 평균 payment 

In [ ]:
query_avg_payment_by_provider = pd.read_sql_query(
    """
    SELECT 
        PRVDR_NUM, 
        AVG(CLM_PMT_AMT) AS avg_payment
    FROM inpatient_claims
    GROUP BY PRVDR_NUM
    ORDER BY avg_payment DESC
    LIMIT 10
    """,
    conn,
)

print(query_avg_payment_by_provider)


  PRVDR_NUM   avg_payment
0     P0120  13101.505014
1     P0115  13003.489802
2     P0119  12884.242536
3     P0117  12739.984766
4     P0116  12493.937052
5     P0118  12281.089940
6     P0090   5725.943187
7     P0012   5455.240137
8     P0054   5447.101733
9     P0072   5417.585109


## 테이블 목록 조회 in claims.db (sqlite_master)

In [ ]:

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    """,
    conn,
)

print(f"the tables in claims.db: \n {tables}")

the tables in claims.db: 
                 name
0   inpatient_claims
1  provider_features


# SQL 4: SELECT, WHERE, GROUP BY, ORDER BY, LIMIT, AGGREGATE

In [ ]:
# SQL 4-1: SELECT, WHERE
clm_high_payment = pd.read_sql_query(
    """
    SELECT
        CLM_ID,
        PRVDR_NUM,
        CLM_PMT_AMT
    FROM inpatient_claims
    WHERE CLM_PMT_AMT > 10000
    ORDER BY CLM_PMT_AMT DESC
    LIMIT 10
    """,
    conn,
)

print(f" To practice where clause by clm_high_payment: \n {clm_high_payment}")

print("--------------------------------------------------------\n")

# SQL 4-2: GROUP BY
#  - 처리순서
#     - 1단계: FROM inpatient_claim (원천테이블)
#     - 2단계: GROUP BY PRVDR_NUM (provider 별로 묶기)
#     - 3단계: SELECT PRVDR_NUM, COUNT(*) AS claim_count (provider 별 claim 수 조회)    
#     - 4단계: ORDER BY claim_count DESC (claim 수 내림차순 정렬)
#     - 5단계: LIMIT 10 (상위 10개 조회)
group_by_provider = pd.read_sql_query(
    """
    SELECT
        PRVDR_NUM,
        COUNT(*) AS claim_count
    FROM inpatient_claims
    GROUP BY PRVDR_NUM
    ORDER BY claim_count DESC
    LIMIT 10
    """,
    conn,
)

print(f" To practice group by clause by group_by_provider: \n {group_by_provider} \n")

print("--------------------------------------------------------\n")



# SQL 4-3: Aggregate

aggregate_by_provider = pd.read_sql_query(
    """
    SELECT
        PRVDR_NUM,
        COUNT(*) AS claim_count,
        COUNT(DISTINCT DESYNPUF_ID)
            AS unique_beneficiaries,
        SUM(CLM_PMT_AMT)
            AS total_payment,
        AVG(CLM_PMT_AMT)
            AS avg_payment,
        MAX(CLM_PMT_AMT)
            AS max_payment
    FROM inpatient_claims
    GROUP BY PRVDR_NUM
    ORDER BY avg_payment DESC
    LIMIT 10
    """,
    conn,
)

print(f" To practice aggregate function by aggregate_by_provider: \n {aggregate_by_provider} \n")


 To practice where clause by clm_high_payment: 
        CLM_ID PRVDR_NUM  CLM_PMT_AMT
0  C000001852     P0117     69247.03
1  C000001203     P0115     64407.39
2  C000009415     P0116     56385.12
3  C000004749     P0116     55906.77
4  C000000321     P0118     55549.46
5  C000007605     P0117     55144.08
6  C000004885     P0116     54488.01
7  C000010794     P0119     52718.47
8  C000011377     P0117     51585.99
9  C000010027     P0118     48963.27
--------------------------------------------------------

 To practice group by clause by group_by_provider: 
   PRVDR_NUM  claim_count
0     P0115          353
1     P0120          351
2     P0119          347
3     P0116          346
4     P0118          331
5     P0117          321
6     P0091          111
7     P0083          106
8     P0055          106
9     P0077          103 

--------------------------------------------------------

 To practice aggregate function by aggregate_by_provider: 
   PRVDR_NUM  claim_count  unique_benef

## SQL 5:  HAVING and WHERE

  - 1. WHERE (집계 전: 개별 행 필터링)
    - 동작 시점: 데이터를 그룹으로 묶기(GROUP BY) 전에 작동합니다.
    - 대상: 원천 데이터 테이블의 개별 행(Row) 하나하나를 검사합니다.
    - 특징: COUNT(), SUM(), AVG() 같은 집계 함수를 조건으로 쓸 수 없슴

  - 2. HAVING (집계 후: 그룹 필터링)
    - 동작 시점: 데이터를 그룹으로 묶은 후(GROUP BY) 작동합니다.
    - 대상: 그룹화된 데이터를 검사합니다.
    - 특징: COUNT(), SUM(), AVG() 같은 집계 함수를 조건으로 쓸 수 있슴

In [ ]:
#  - 5.1 예시: HAVING 사용
#  - 처리순서
#     - 1단계: FROM inpatient_claim (원천테이블)
#     - 2단계: GROUP BY PRVDR_NUM (provider 별로 묶기)
#     - 3단계: SELECT PRVDR_NUM, COUNT(*) AS claim_count (provider 별 claim 수 조회)SELECT
#     - 4단계: HAVING COUNT(*) >= 150 (provider 별 claim 수 조건 필터링)
#     - 5단계: ORDER BY claim_count DESC (claim 수 내림차순 정렬)
having_example = pd.read_sql_query(
    """
    SELECT
        PRVDR_NUM,
        COUNT(*) AS claim_count,
        AVG(CLM_PMT_AMT) AS avg_payment
    FROM inpatient_claims
    GROUP BY PRVDR_NUM
    HAVING COUNT(*) >= 150
    ORDER BY claim_count DESC
    """,
    conn,
)

print(f" To practice having clause by having_example: \n {having_example} \n")

 To practice having clause by having_example: 
   PRVDR_NUM  claim_count   avg_payment
0     P0115          353  13003.489802
1     P0120          351  13101.505014
2     P0119          347  12884.242536
3     P0116          346  12493.937052
4     P0118          331  12281.089940
5     P0117          321  12739.984766 



In [ ]:
# 5.2 예시: WHERE 와 HAVING 같이 사용
# - 처리순서
#     - 1단계: FROM inpatient_claim (원천테이블) 
#     - 2단계: WHERE CLM_PMT_AMT >= 5000 (claim 금액 조건 필터링)
#     - 3단계: GROUP BY PRVDR_NUM (provider 별로 묶기)
#     - 4단계: SELECT PRVDR_NUM, COUNT(*) AS high_payment_claim_count (provider 별 claim 수 조회)
#     - 5단계: HAVING COUNT(*) >= 30 (provider 별 claim 수 조건 필터링)
#     - 6단계: ORDER BY high_payment_claim_count DESC (claim 수 내림차순 정렬)
having_where_example = pd.read_sql_query(
    """
    SELECT
        PRVDR_NUM,
        COUNT(*) AS high_payment_claim_count,
        AVG(CLM_PMT_AMT) AS avg_payment
    FROM inpatient_claims
    WHERE CLM_PMT_AMT >= 5000
    GROUP BY PRVDR_NUM
    HAVING COUNT(*) >= 30
    ORDER BY high_payment_claim_count DESC
    """,
    conn,
)

print(f" To practice having and where clause by having_where_example: \n {having_where_example} \n")

 To practice having and where clause by having_where_example: 
    PRVDR_NUM  high_payment_claim_count   avg_payment
0      P0115                       294  15070.194286
1      P0120                       292  15114.064932
2      P0119                       291  14734.987320
3      P0116                       283  14525.801237
4      P0117                       268  14614.133769
..       ...                       ...           ...
89     P0065                        30   7923.693333
90     P0050                        30   7712.223333
91     P0019                        30   8315.327333
92     P0014                        30   6861.315000
93     P0002                        30   8293.286333

[94 rows x 3 columns] 



## SQL 6: CASE
 - payment를 세 그룹으로 나누어 보기

In [52]:
case_example = pd.read_sql_query(
    """
    SELECT
        CLM_ID,
        PRVDR_NUM,
        CLM_PMT_AMT,
        CASE
            WHEN CLM_PMT_AMT >= 15000
                THEN 'HIGH'
            WHEN CLM_PMT_AMT >= 5000
                THEN 'MEDIUM'
            ELSE 'LOW'
        END AS payment_band
    FROM inpatient_claims
    ORDER BY CLM_PMT_AMT DESC
    LIMIT 20
    """,
    conn,
)

print(f" To practice case clause by case_example: \n {case_example} \n")


 To practice case clause by case_example: 
         CLM_ID PRVDR_NUM  CLM_PMT_AMT payment_band
0   C000001852     P0117     69247.03         HIGH
1   C000001203     P0115     64407.39         HIGH
2   C000009415     P0116     56385.12         HIGH
3   C000004749     P0116     55906.77         HIGH
4   C000000321     P0118     55549.46         HIGH
5   C000007605     P0117     55144.08         HIGH
6   C000004885     P0116     54488.01         HIGH
7   C000010794     P0119     52718.47         HIGH
8   C000011377     P0117     51585.99         HIGH
9   C000010027     P0118     48963.27         HIGH
10  C000005993     P0119     48688.78         HIGH
11  C000003072     P0119     48603.13         HIGH
12  C000008422     P0115     48263.69         HIGH
13  C000009864     P0118     48239.40         HIGH
14  C000000850     P0119     48201.62         HIGH
15  C000011814     P0120     46436.69         HIGH
16  C000004610     P0120     46401.31         HIGH
17  C000005427     P0119     45287.54 

## SQL 7: CASE + GROUP BY
- provider가 아니라 payment band 자체를 aggregate
- Below code logic description: 
    - raw continuous payment -> CASE -> categorical representation(HIGH/MEDIUM/LOW) -> GROUP BY -> 각 category summary

In [ ]:
# 처리순서
#     - 1단계: FROM inpatient_claim (원천테이블)
#     - 2단계: GROUP BY CASE WHEN ... END
#     - 3단계: SELECT CASE ... AS payment_band, COUNT(*) AS claim_count, AVG(...)

case_group_example = pd.read_sql_query(
    """
    SELECT
        CASE
            WHEN CLM_PMT_AMT >= 15000
                THEN 'HIGH'
            WHEN CLM_PMT_AMT >= 5000
                THEN 'MEDIUM'
            ELSE 'LOW'
        END AS payment_band,
        COUNT(*) AS claim_count,
        AVG(CLM_PMT_AMT) AS avg_payment
    FROM inpatient_claims
    GROUP BY
        CASE
            WHEN CLM_PMT_AMT >= 15000
                THEN 'HIGH'
            WHEN CLM_PMT_AMT >= 5000
                THEN 'MEDIUM'
            ELSE 'LOW'
        END
    """,
    conn,
)

print(f" To practice case + group by clause by case_group_example: \n {case_group_example} \n")

 To practice case + group by clause by case_group_example: 
   payment_band  claim_count   avg_payment
0         HIGH          759  22605.205007
1          LOW         6470   2788.361665
2       MEDIUM         4771   8094.088751 



## SQL 8: CTE (Common Table Expression)
- Intuition: 복잡한 query 중간 결과에 이름을 붙여서 다음 query에서 사용
- Structure

        ```sql
        WITH provider_summary AS (
            ...
        )
        SELECT ...
        FROM provider_summary
        ```

- Explanation:
    - Step A: raw claims → provider summary
    - Step B: provider summary → filtering/sorting


In [54]:
cte_example = pd.read_sql_query(
    """
    WITH provider_summary AS (
        SELECT
            PRVDR_NUM,
            COUNT(*) AS claim_count,
            COUNT(DISTINCT DESYNPUF_ID)
                AS unique_beneficiaries,
            AVG(CLM_PMT_AMT)
                AS avg_payment,
            SUM(CLM_PMT_AMT)
                AS total_payment
        FROM inpatient_claims
        GROUP BY PRVDR_NUM
    )

    SELECT *
    FROM provider_summary
    WHERE avg_payment > 7000
    ORDER BY avg_payment DESC
    """,
    conn,
)

print(f" To practice CTE by cte_example: \n {cte_example} \n")

 To practice CTE by cte_example: 
   PRVDR_NUM  claim_count  unique_beneficiaries   avg_payment  total_payment
0     P0120          351                   340  13101.505014     4598628.26
1     P0115          353                   337  13003.489802     4590231.90
2     P0119          347                   337  12884.242536     4470832.16
3     P0117          321                   307  12739.984766     4089535.11
4     P0116          346                   340  12493.937052     4322902.22
5     P0118          331                   316  12281.089940     4065040.77 



## SQL 9: JOIN
- left join: left table 보존, right table 교집합 부분 연결
- inner join: INNER JOIN은 JOIN을 기준으로 왼쪽과 오른쪽 테이블 모두에 공통으로 존재하는(일치하는) 데이터만 남기는 결합 방식

In [ ]:

# left table: inpatient_claims(=c), right table: provider_features(=p)
# 왜 이런 아래와 같은 작업을 할까: 이 특정 청구건(CLM_ID)이 사기인가?"를 판단하려면, 단일 청구 금액(CLM_PMT_AMT)만 보는 것보다 "이 청구를 올린 병원이 평소에 고액 청구를 얼마나 자주 하는 곳인가?(high_cost_claim_share)" 같은 병원 단위의 문맥(Context) 정보를 함께 학습시키는 것이 훨씬 유리하기 때문

join_example = pd.read_sql_query(
    """
    SELECT
        c.CLM_ID,
        c.PRVDR_NUM,
        c.CLM_PMT_AMT,
        p.claim_count,
        p.avg_payment,
        p.high_cost_claim_share
    FROM inpatient_claims AS c
    LEFT JOIN provider_features AS p
        ON c.PRVDR_NUM = p.provider_id
    LIMIT 20
    """,
    conn,
)

print(f" To practice join clause by join_example: \n {join_example} \n")

 To practice join clause by join_example: 
         CLM_ID PRVDR_NUM  CLM_PMT_AMT  claim_count   avg_payment  \
0   C000000001     P0107      4500.44           90   4745.429778   
1   C000000002     P0061     12500.86           90   4734.912444   
2   C000000003     P0116     12857.23          346  12493.937052   
3   C000000004     P0097      4593.34           79   4685.637722   
4   C000000005     P0013     10138.87           76   5313.513947   
5   C000000006     P0120      2949.43          351  13101.505014   
6   C000000007     P0106      1837.69           75   4442.523200   
7   C000000008     P0109      1818.61           92   4524.004022   
8   C000000009     P0018      8857.23           95   4189.728632   
9   C000000010     P0063      4881.21           96   4566.537292   
10  C000000011     P0052     14204.06           78   4723.010385   
11  C000000012     P0118     12470.38          331  12281.089940   
12  C000000013     P0089      4526.68           85   4635.299647   
13  

## SQL 10: Window Function
- Below Code: claim row는 유지하면서 그 claim의 provider의 평균 payment를 옆에 추가

In [ ]:
#    PARTITION BY PRVDR_NUM: provider 별로 따로 계산하라
window_example = pd.read_sql_query(
    """
    SELECT
        PRVDR_NUM,
        CLM_ID,
        CLM_PMT_AMT,
        AVG(CLM_PMT_AMT) OVER (
            PARTITION BY PRVDR_NUM
        ) AS provider_avg_payment
    FROM inpatient_claims
    LIMIT 20
    """,
    conn,
)

print(f" To practice window function by window_example: \n {window_example} \n")



 To practice window function by window_example: 
    PRVDR_NUM      CLM_ID  CLM_PMT_AMT  provider_avg_payment
0      P0001  C000000423      4131.28           4609.138065
1      P0001  C000000647      2103.28           4609.138065
2      P0001  C000000741      8988.28           4609.138065
3      P0001  C000000744      6012.80           4609.138065
4      P0001  C000000775      2426.61           4609.138065
5      P0001  C000000811      9909.34           4609.138065
6      P0001  C000000998      1529.25           4609.138065
7      P0001  C000001158      1514.50           4609.138065
8      P0001  C000001221      7267.72           4609.138065
9      P0001  C000001314      2685.56           4609.138065
10     P0001  C000001323      5465.80           4609.138065
11     P0001  C000001617      6325.60           4609.138065
12     P0001  C000001837      6343.21           4609.138065
13     P0001  C000002299     11786.17           4609.138065
14     P0001  C000002585      8809.73           46

## SQL 11: ROW NUMBER: window function의 예

In [ ]:
# provider 내부에서 payment가 큰 순서대로 번호를 붙이라
# PARTITION BY PRVDR_NUM: provider 별로 따로 계산하라

row_number_example = pd.read_sql_query(
    """
    SELECT
        PRVDR_NUM,
        CLM_ID,
        CLM_PMT_AMT,
        ROW_NUMBER() OVER (
            PARTITION BY PRVDR_NUM                               
            ORDER BY CLM_PMT_AMT DESC
        ) AS payment_row_number
    FROM inpatient_claims
    ORDER BY
        PRVDR_NUM,
        payment_row_number
    LIMIT 30
    """,
    conn,
)

print(f" To practice row number by row_number_example: \n {row_number_example} \n")



 To practice row number by row_number_example: 
    PRVDR_NUM      CLM_ID  CLM_PMT_AMT  payment_row_number
0      P0001  C000005178     16686.70                   1
1      P0001  C000002299     11786.17                   2
2      P0001  C000009818     11608.87                   3
3      P0001  C000006120     11343.39                   4
4      P0001  C000002731     10091.18                   5
..       ...         ...          ...                 ...
95     P0002  C000003218     11986.37                   3
96     P0002  C000005625     11750.12                   4
97     P0002  C000009617     11074.82                   5
98     P0002  C000002384     11036.69                   6
99     P0002  C000011318     10806.72                   7

[100 rows x 4 columns] 



## SQL 12: RANK (window function의 한 예)

In [ ]:
# ROW_NUMBER와 RANK 차이는 tie가 있을 때 나타남
# 예시: 1, 2, 2, 4 중에서 1등이 2명이면, ROW_NUMBER는 1, 2, 3, 4이지만, RANK는 1, 2, 2, 4

rank_example = pd.read_sql_query(
    """
    SELECT
        PRVDR_NUM,
        CLM_ID,
        CLM_PMT_AMT,
        RANK() OVER (
            PARTITION BY PRVDR_NUM
            ORDER BY CLM_PMT_AMT DESC
        ) AS payment_rank
    FROM inpatient_claims
    ORDER BY
        PRVDR_NUM,
        payment_rank
    LIMIT 30
    """,
    conn,
)

print(f" To practice rank by rank_example: \n {rank_example} \n")

 To practice rank by rank_example: 
    PRVDR_NUM      CLM_ID  CLM_PMT_AMT  payment_rank
0      P0001  C000005178     16686.70             1
1      P0001  C000002299     11786.17             2
2      P0001  C000009818     11608.87             3
3      P0001  C000006120     11343.39             4
4      P0001  C000002731     10091.18             5
5      P0001  C000000811      9909.34             6
6      P0001  C000003503      9528.02             7
7      P0001  C000003492      9254.31             8
8      P0001  C000000741      8988.28             9
9      P0001  C000002585      8809.73            10
10     P0001  C000007031      8731.92            11
11     P0001  C000006623      8256.55            12
12     P0001  C000003038      8083.73            13
13     P0001  C000004991      7946.02            14
14     P0001  C000010242      7323.99            15
15     P0001  C000001221      7267.72            16
16     P0001  C000008584      6847.28            17
17     P0001  C000007176   

## SQL 13: Top 3 claim per provider

In [ ]:
# CTE + window function
# 각 provider 별로 top 3개씩 뽑아오기
top_3_claims_example = pd.read_sql_query(
    """
    WITH ranked_claims AS (
        SELECT
            PRVDR_NUM,
            CLM_ID,
            CLM_PMT_AMT,
            ROW_NUMBER() OVER (
                PARTITION BY PRVDR_NUM
                ORDER BY CLM_PMT_AMT DESC
            ) AS rn
        FROM inpatient_claims
    )

    SELECT
        PRVDR_NUM,
        CLM_ID,
        CLM_PMT_AMT,
        rn
    FROM ranked_claims
    WHERE rn <= 3
    ORDER BY
        PRVDR_NUM,
        rn
    LIMIT 30
    """,
    conn,
)

print(f" To practice top 3 claims per provider by top_3_claims_example: \n {top_3_claims_example} \n")

 To practice top 3 claims per provider by top_3_claims_example: 
    PRVDR_NUM      CLM_ID  CLM_PMT_AMT  rn
0      P0001  C000005178     16686.70   1
1      P0001  C000002299     11786.17   2
2      P0001  C000009818     11608.87   3
3      P0002  C000010147     17450.82   1
4      P0002  C000009305     12236.34   2
5      P0002  C000003218     11986.37   3
6      P0003  C000000683     13382.25   1
7      P0003  C000002571     12186.26   2
8      P0003  C000007488     10316.82   3
9      P0004  C000002031     21402.38   1
10     P0004  C000001205     15451.48   2
11     P0004  C000011446     13308.42   3
12     P0005  C000001161     14080.77   1
13     P0005  C000006824     13798.21   2
14     P0005  C000002560     13228.93   3
15     P0006  C000003886     12171.29   1
16     P0006  C000006868     11480.03   2
17     P0006  C000001620      9847.65   3
18     P0007  C000010013     23548.61   1
19     P0007  C000003595     17332.07   2
20     P0007  C000011725     11789.68   3
21     P00